# Merging individual CSV files per Google page into a single CSV per country

In [ ]:
import pandas as pd
import glob
import os

# Path to the folder containing CSV files
ruta_carpeta = "data-international-searches/links/links_usa"  # Adjust for each country

# Output path within the project
ruta_salida = os.path.join("data-international-searches/links", "links_usa.csv")  # Adjust for each country

# Search for all CSV files in the folder
archivos_csv = glob.glob(os.path.join(ruta_carpeta, "*.csv"))

# List to store DataFrames
dataframes = []

for archivo in archivos_csv:
    df = pd.read_csv(archivo)
    dataframes.append(df)

# Merge all DataFrames
df_final = pd.concat(dataframes, ignore_index=True)

# Remove duplicates if necessary
df_final.drop_duplicates(inplace=True)

# Save merged CSV
df_final.to_csv(ruta_salida, index=False)

print(f"Merged CSV saved at: {ruta_salida}")


# Add "country" column to each country CSV

In [ ]:
import pandas as pd
import os

# Configuration
nombre_archivo_original = "links_usa.csv"  # Adjust for each country
valor_country = "usa"  # Adjust for each country

# Input path
ruta_entrada = os.path.join("data-international-searches/links", nombre_archivo_original)

# Load CSV
df = pd.read_csv(ruta_entrada)

# Add country column
df["country"] = valor_country

# Output file name
nombre_base = os.path.splitext(nombre_archivo_original)[0]
nuevo_nombre_archivo = f"{nombre_base}_withcountrycolumn.csv"

# Output folder
carpeta_salida = "data-international-searches/linkswithcountrycolumn"
os.makedirs(carpeta_salida, exist_ok=True)

# Output path
ruta_salida = os.path.join(carpeta_salida, nuevo_nombre_archivo)

# Save CSV
df.to_csv(ruta_salida, index=False)

print(f"New CSV saved at: {ruta_salida}")


# Merge all CSVs with "country" column into a single dataset

In [ ]:
import pandas as pd
import os

# Input folder
carpeta_entrada = "data-international-searches/linkswithcountrycolumn"

# Output folder
carpeta_salida = "data-international-searches/linksall"
os.makedirs(carpeta_salida, exist_ok=True)

# List to store DataFrames
dfs = []

for archivo in os.listdir(carpeta_entrada):
    if archivo.endswith(".csv"):
        ruta = os.path.join(carpeta_entrada, archivo)
        df = pd.read_csv(ruta)
        dfs.append(df)

# Merge all DataFrames
df_total = pd.concat(dfs, ignore_index=True)

# Save output
archivo_salida = os.path.join(carpeta_salida, "links_all.csv")
df_total.to_csv(archivo_salida, index=False)

print(f"Merged file saved at: {archivo_salida}")


# Add "httpcode" column and retrieve HTTP status for each link

In [ ]:
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import os

# Paths
archivo_entrada = "data-international-searches/linksall/links_all.csv"
carpeta_salida = "data-international-searches/linksallwithhttpcode"
archivo_salida = os.path.join(carpeta_salida, "links_all_withhttpcode.csv")

os.makedirs(carpeta_salida, exist_ok=True)

# Load CSV
df = pd.read_csv(archivo_entrada)

def obtener_http_code(url):
    try:
        response = requests.head(url, allow_redirects=True, timeout=5, headers={"User-Agent": "Mozilla/5.0"})
        return response.status_code
    except requests.RequestException:
        return None

urls = df["Link"].tolist()
http_codes = [None] * len(urls)

max_workers = 20
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    future_to_index = {executor.submit(obtener_http_code, url): i for i, url in enumerate(urls)}

    for future in tqdm(as_completed(future_to_index), total=len(urls), desc="Fetching HTTP codes"):
        i = future_to_index[future]
        try:
            http_codes[i] = future.result()
        except Exception:
            http_codes[i] = None

df["httpcode"] = http_codes

df.to_csv(archivo_salida, index=False)

print(f"Process completed. File saved at: {archivo_salida}")


# Filter links by uniqueness and HTTP 200 status

In [ ]:
import pandas as pd
import os

archivo_entrada = "data-international-searches/linksallwithhttpcode/links_all_withhttpcode.csv"
carpeta_salida = "data-international-searches/linksallwithhttpcodeonlyunique200"
archivo_salida = os.path.join(carpeta_salida, "links_all_withhttpcode_onlyunique200.csv")

os.makedirs(carpeta_salida, exist_ok=True)

df = pd.read_csv(archivo_entrada)

df = df.drop_duplicates(subset="Link", keep="first")
df = df[df["httpcode"].isin([200, 200.0])]

df.to_csv(archivo_salida, index=False)

print(f"Filtered file saved at: {archivo_salida}")
print(f"Total links after filtering: {len(df)}")
